In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib

# Charger le modèle et le scaler
model = load_model('model_bigru.keras')
scaler = joblib.load('scaler_minmax_gru.pkl')

In [ ]:
def predict_future_prices(stock_symbol, days_to_predict=7):
    # Charger les données historiques
    data = pd.read_csv(f'stock/{stock_symbol}.csv')
    close_prices = data['Close'].values.reshape(-1, 1)
    
    # Normaliser les données
    scaled_data = scaler.transform(close_prices)
    
    # Préparer les séquences pour la prédiction
    sequence_length = 3
    future_prices = []
    current_sequence = scaled_data[-sequence_length:]
    
    # Prédire les prix pour les prochains jours
    for _ in range(days_to_predict):
        # Préparer la séquence
        X = current_sequence.reshape(1, sequence_length, 1)
        
        # Faire la prédiction
        next_price_scaled = model.predict(X, verbose=0)
        
        # Ajouter la prédiction aux résultats
        future_prices.append(next_price_scaled[0][0])
        
        # Mettre à jour la séquence pour la prochaine prédiction
        current_sequence = np.append(current_sequence[1:], next_price_scaled)
    
    # Convertir les prédictions à l'échelle originale
    future_prices = np.array(future_prices).reshape(-1, 1)
    future_prices = scaler.inverse_transform(future_prices)
    
    return future_prices.flatten()

In [ ]:
# Liste des actions
stocks = [f.split('.')[0] for f in pd.read_csv('predictions/prix_bigru.csv').columns[1:]]

# Créer un DataFrame pour stocker les prédictions
future_dates = pd.date_range(start=pd.Timestamp.now(), periods=7, freq='B')
predictions_df = pd.DataFrame(index=future_dates)

# Prédire pour chaque action
for stock in stocks:
    print(f"\nPrédictions pour {stock}:")
    future_prices = predict_future_prices(stock)
    predictions_df[stock] = future_prices
    
    # Afficher le dernier prix connu et les prédictions
    current_price = pd.read_csv(f'stock/{stock}.csv')['Close'].iloc[-1]
    print(f"Prix actuel: {current_price:.2f}")
    
    print("\nPrédictions pour les 7 prochains jours ouvrables:")
    for day, price in zip(future_dates, future_prices):
        change = ((price - current_price) / current_price) * 100
        print(f"{day.strftime('%Y-%m-%d')}: {price:.2f} (Variation: {change:+.2f}%)")    

In [ ]:
# Sauvegarder les prédictions
predictions_df.to_csv('predictions/future_predictions.csv')
print("\nLes prédictions ont été sauvegardées dans 'predictions/future_predictions.csv'")

# Afficher le tableau des prédictions
print("\nTableau des prédictions:")
display(predictions_df)